<a href="https://colab.research.google.com/github/Pujitha-pitta/Flyrank-ML-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*


I frame my lane as a binary classification task.

The task is to predict whether an existing content item should be prioritised for a content refresh.

The model output would contain two classes:

1 = prioritise the content item for refresh
0 = do not prioritise it yet

Classification is suitable because the intended output is a decision-support category rather than an exact numerical prediction.

The model would help a content team identify which existing pages should be reviewed first when editorial time is limited.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*



The proposed target is needs_refresh_proxy.

The dataset does not contain a directly observed label stating whether an editor decided to refresh a content item. Therefore, I define an initial proxy using the observed traffic trend:

1 = the content item has a downward trend
0 = the content item is stable or improving

The proxy is created from the existing trend_direction column.

A downward trend does not automatically prove that a content item needs rewriting. It is a directional signal that suggests the item may deserve human review.

The target therefore supports content prioritisation rather than making an automatic publishing decision.

In [9]:
df_task = df.copy()

df_task["needs_refresh_proxy"] = (
    df_task["trend_direction"].eq("down")
).astype(int)

print("Target distribution:")
print(df_task["needs_refresh_proxy"].value_counts())

print("\nTarget percentages:")
print(
    df_task["needs_refresh_proxy"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

display(
    df_task[
        [
            "content_id",
            "trend_direction",
            "trend_pct",
            "needs_refresh_proxy"
        ]
    ].head(10)
)

Target distribution:
needs_refresh_proxy
1    16262
0    13738
Name: count, dtype: int64

Target percentages:
needs_refresh_proxy
1    54.21
0    45.79
Name: proportion, dtype: float64


,content_id,trend_direction,trend_pct,needs_refresh_proxy
0,content_304f48230142,down,-41.4,1
1,content_a1fb4e703a9e,down,-57.7,1
2,content_9aa793d4d895,down,-60.9,1
3,content_331d6c4de07b,stable,-13.8,0
4,content_d99b7a2d90ca,down,-34.7,1
5,content_d4084a4bc775,down,-38.9,1
6,content_9a34b442b552,down,-92.3,1
7,content_a63219c6e95a,stable,0.6,0
8,content_5e6c160719bc,down,-58.8,1
9,content_c27558df2b0c,down,-29.2,1




Because trend_direction is used to construct the proxy target, it must not be included as a model feature.

I would also exclude trend_pct because it directly represents the same recent performance trend and would make the prediction unrealistically easy.

The model should instead learn from information available before the refresh decision, such as content age, days since update, search demand, competition, content type, historical impressions, position, engagement, and previous-period performance.

## 3. Success metric

*One metric you can defend. What number means 'good'?*



My primary success metric is the F1-score for the refresh-priority class.

Precision measures how many content items predicted for refresh are actually part of the observed declining group.

Recall measures how many declining content items the model successfully identifies.

Both errors matter:

- Low precision would waste editorial time on unnecessary refreshes.
- Low recall would cause the team to miss content that is losing performance.

F1-score provides one balanced measure of precision and recall.

For this initial analysis, I would consider an F1-score of 0.70 or higher on unseen validation data to be a useful directional result, provided that it also improves meaningfully over a simple baseline.

This is an exploratory decision-support benchmark, not a production guarantee.

In [2]:
!git clone https://github.com/Pujitha-pitta/Flyrank-ML-internship.git

Cloning into 'Flyrank-ML-internship'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 133 (delta 43), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (133/133), 1.85 MiB | 25.65 MiB/s, done.
Resolving deltas: 100% (43/43), done.


In [3]:
%cd Flyrank-ML-internship

/content/Flyrank-ML-internship


In [4]:
!find . -maxdepth 3 -type f

./scripts/run_all.py
./scripts/05_build_pdf_report.py
./scripts/02_baseline_score.py
./scripts/03_train_model.py
./scripts/04_evaluate_and_export.py
./scripts/01_prepare_features.py
./scripts/ml_utils.py
./AGENTS.md
./requirements.txt
./data/raw/content_refresh_anonymized.csv
./.gitignore
./README.md
./submission/README.md
./submission/paper_url.txt
./CLAUDE.md
./.github/workflows/data-path-smoke.yml
./.github/workflows/smoke-test.yml
./work/README.md
./work/capstone_report_template.md
./work/notebooks/w04_signal_audit.ipynb
./work/notebooks/w03_data_contract.ipynb
./work/notebooks/w02_ml_task_framing.ipynb
./work/notebooks/w06_validation_audit.ipynb
./work/notebooks/w07_action_playbook.ipynb
./work/notebooks/capstone.ipynb
./work/notebooks/w04_baseline_score.ipynb
./work/notebooks/w05_model.ipynb
./work/notebooks/w01_research_question.ipynb
./work/notebooks/w03_feature_leakage_check.ipynb
./LICENSE
./DATA_USE.md
./notebooks/03_working_with_the_full_release.ipynb
./notebooks/01_first_l

In [5]:
!cat skills/README.md

# Skills — the router

This folder is a small library of **skills**: focused instruction files your AI assistant loads
one at a time. One skill per task keeps the assistant sharp — its context window is small, and
filling it with everything makes it worse at the one thing you need.

**How to use it (repo-reading agents — Claude Code, Cursor, Codex):** they find this file
automatically via `AGENTS.md` / `CLAUDE.md`. Just tell your assistant which task you're doing.

**Using a chat-only assistant (ChatGPT / Gemini in a browser)?** Open the skill file on GitHub,
copy its whole content, and paste it into your chat before asking for help. That's it.

## The table — find your task, load ONE skill

| Your task | Load this skill | Also load for data work |
|---|---|---|
| Any task — how to work with your assistant at all | `directing-your-ai-assistant/SKILL.md` | — |
| Pick a lane, frame your question (ML-02, ML-03) | `framing-ml-problems/SKILL.md` | `flyrank/flyrank-data/SKILL.md` |
| Write +

In [6]:
from pathlib import Path

repo_root = Path.cwd()

candidate_files = sorted(
    list(repo_root.rglob("*.csv")) +
    list(repo_root.rglob("*.parquet")) +
    list(repo_root.rglob("*.json"))
)

print("Possible repository data files:\n")

for file in candidate_files:
    print(file.relative_to(repo_root))

Possible repository data files:

data/raw/content_refresh_anonymized.csv
outputs/refresh_queue_sample.csv


In [8]:
from pathlib import Path
import pandas as pd

repo_root = Path.cwd()

data_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"

print("Looking for:", data_path)

df = pd.read_csv(data_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())

Looking for: /content/Flyrank-ML-internship/data/raw/content_refresh_anonymized.csv
Shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*



One row represents one anonymised content item.

Each row contains information about that content item's characteristics, search opportunity, age, historical visibility, engagement, position, traffic trend, and update history.

This is the correct unit of analysis because the final action is also made at the content-item level: deciding whether a specific item should be prioritised for editorial review and possible refresh.

In [10]:
unit_columns = [
    "content_id",
    "search_volume",
    "competition_level",
    "content_type",
    "main_intent",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "trend_direction",
    "needs_refresh_proxy"
]

display(df_task[unit_columns].head(10))

,content_id,search_volume,competition_level,content_type,main_intent,word_count,impressions_90d,clicks_90d,sessions_90d,content_age_days,days_since_last_update,ctr,avg_position,engagement_rate,trend_direction,needs_refresh_proxy
0,content_304f48230142,10.0,HIGH,keyword article,transactional,3221.0,3803,29,17,187,20,0.76,10.6,5.88,down,1
1,content_a1fb4e703a9e,90.0,LOW,keyword article,informational,2481.0,15320,7,9,445,25,0.05,20.3,0.00,down,1
2,content_9aa793d4d895,0.0,LOW,keyword article,informational,3515.0,12581,11,11,141,20,0.09,36.5,0.00,down,1
3,content_331d6c4de07b,10.0,LOW,keyword article,commercial,NaN,11751,58,78,463,22,0.49,6.2,1.28,stable,0
4,content_d99b7a2d90ca,0.0,LOW,keyword article,informational,2803.0,19140,24,145,263,14,0.13,44.0,0.00,down,1
5,content_d4084a4bc775,720.0,HIGH,keyword article,transactional,3080.0,3970,1,5,147,20,0.03,8.5,0.00,down,1
6,content_9a34b442b552,0.0,LOW,keyword article,informational,3059.0,20,0,1,90,20,0.00,7.0,0.00,down,1
7,content_a63219c6e95a,590.0,MEDIUM,keyword article,commercial,NaN,1724,1,28,445,22,0.06,21.2,3.57,stable,0
8,content_5e6c160719bc,0.0,LOW,keyword article,informational,3807.0,32574,29,68,90,20,0.09,46.0,5.88,down,1
9,content_c27558df2b0c,0.0,LOW,keyword article,informational,NaN,1240,2,3,257,104,0.16,4.9,0.00,down,1


The dataframe contains 30,000 observations. Each observation is an individual content item, identified by an anonymised content_id.

Client identifiers are not needed for this task framing and will not be displayed or used as evidence in the notebook.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*



A fixed rule might say:

> Refresh every content item that has not been updated for more than 180 days.

That rule would be too limited because age alone does not determine whether a content item is losing value.

Refresh priority may depend on several interacting factors, including:

- Days since the last update
- Content age
- Search volume
- Competition
- Previous impressions and clicks
- Average search position
- Click-through rate
- Engagement rate
- Content type
- Search intent

For example, an old page may still perform well and require no immediate action, while a recently published page may already be declining because of strong competition or poor engagement.

These relationships are conditional and may not be captured by one threshold or if-statement.

Machine learning can measure patterns across several features and estimate which content items resemble previously observed declining items.

The output would support a real content action: ranking or filtering content items for human review and possible refresh. It would not automatically rewrite, delete, or publish content.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.